<a href="https://colab.research.google.com/github/Dharshini1701/priyadharshini/blob/main/Real_time_market_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded=files.upload()

Saving Real_Time_Algorithmic_Market_Intelligence_Raw.csv to Real_Time_Algorithmic_Market_Intelligence_Raw.csv


In [ ]:
import pandas as pd
df=pd.read_csv("Real_Time_Algorithmic_Market_Intelligence_Raw.csv")

print("Rows:",df.shape[0])
print("columns:",df.shape[1])

Rows: 50050
columns: 15


In [ ]:
print("Missing values:",df.isnull().sum())

Missing values: Stock_ID            0
Stock_Symbol        0
Company_Name      120
Sector             80
Market              0
Trade_Date         60
Open_Price         76
High_Price          0
Low_Price           0
Close_Price        65
Volume             90
Market_Cap          0
Daily_Return        0
Volatility          0
Trading_Signal      0
dtype: int64


In [ ]:
print("Duplicated values:",df.duplicated().sum())

Duplicated values: 50


In [ ]:
df = df.drop_duplicates()

print("Rows after duplicate removal:", len(df))

Rows after duplicate removal: 50000


In [ ]:
text_columns = ['Stock_Symbol', 'Company_Name', 'Sector', 'Market', 'Trading_Signal']

for col in text_columns:
    df[col] = df[col].str.strip()

In [ ]:
df['Market'] = df['Market'].str.upper()
df['Market'].value_counts()

,count
Market,
NSE,34990
BSE,15010


In [ ]:
df['Trading_Signal'] = df['Trading_Signal'].str.upper()
df['Trading_Signal'].value_counts()

,count
Trading_Signal,
HOLD,35163
BUY,7446
SELL,7391


In [ ]:
df['Trade_Date'] = pd.to_datetime(df['Trade_Date'], errors='coerce')
df['Trade_Date'].isna().sum()

np.int64(60)

In [ ]:
df = df.dropna(subset=['Trade_Date'])

print("Rows:",df.shape[0])
print("columns",df.shape[1])

Rows: 49940
columns 15


In [ ]:
numeric_columns = [
    'Open_Price',
    'High_Price',
    'Low_Price',
    'Close_Price',
    'Volume',
    'Market_Cap',
    'Daily_Return',
    'Volatility'
]

df[numeric_columns].isnull().sum()

,0
Open_Price,75
High_Price,0
Low_Price,0
Close_Price,65
Volume,90
Market_Cap,0
Daily_Return,0
Volatility,0


In [ ]:
for col in ['Open_Price', 'Close_Price', 'Volume']:
    df[col] = df[col].fillna(df[col].median())

df[['Open_Price', 'Close_Price', 'Volume']].isnull().sum()

,0
Open_Price,0
Close_Price,0
Volume,0


In [ ]:
df['Company_Name'] = df['Company_Name'].fillna('Unknown')
df['Sector'] = df['Sector'].fillna('Unknown')

df[['Company_Name', 'Sector']].isnull().sum()

,0
Company_Name,0
Sector,0


In [ ]:
print("Missing values:",df.isnull().sum())

Missing values: Stock_ID          0
Stock_Symbol      0
Company_Name      0
Sector            0
Market            0
Trade_Date        0
Open_Price        0
High_Price        0
Low_Price         0
Close_Price       0
Volume            0
Market_Cap        0
Daily_Return      0
Volatility        0
Trading_Signal    0
dtype: int64


In [ ]:
print("Rows:",df.shape[0])
print("columns:",df.shape[1])

Rows: 49940
columns: 15


In [ ]:
df.to_csv("Real_Time_Algorithmic_Market_Intelligence_Clean.csv", index=False)

In [ ]:
import sqlite3

conn = sqlite3.connect('market_intelligence.db')

df.to_sql('Real_Time_Algorithmic_Market_Intelligence_Clean',conn,if_exists='replace',index=False)

print("Data loaded into SQLite successfully")

Data loaded into SQLite successfully


In [ ]:
query = """
PRAGMA table_info(Real_Time_Algorithmic_Market_Intelligence_Clean);
"""

result = pd.read_sql_query(query, conn)
print(result[['name', 'type']])

              name       type
0         Stock_ID    INTEGER
1     Stock_Symbol       TEXT
2     Company_Name       TEXT
3           Sector       TEXT
4           Market       TEXT
5       Trade_Date  TIMESTAMP
6       Open_Price       REAL
7       High_Price       REAL
8        Low_Price       REAL
9      Close_Price       REAL
10          Volume       REAL
11      Market_Cap       REAL
12    Daily_Return       REAL
13      Volatility       REAL
14  Trading_Signal       TEXT


In [ ]:
query = """
SELECT
    Market,
    Trading_Signal,
    COUNT(*) AS total_records
FROM Real_Time_Algorithmic_Market_Intelligence_Clean
GROUP BY Market, Trading_Signal
ORDER BY Market, Trading_Signal;
"""

result = pd.read_sql_query(query, conn)
print(result)

  Market Trading_Signal  total_records
0    BSE            BUY           2229
1    BSE           HOLD          10532
2    BSE           SELL           2227
3    NSE            BUY           5206
4    NSE           HOLD          24588
5    NSE           SELL           5158


In [ ]:
query = """
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN Company_Name IS NULL OR TRIM(Company_Name) = '' THEN 1 ELSE 0 END) AS missing_company,
    SUM(CASE WHEN Sector IS NULL OR TRIM(Sector) = '' THEN 1 ELSE 0 END) AS missing_sector
FROM Real_Time_Algorithmic_Market_Intelligence_Clean;
"""

result = pd.read_sql_query(query, conn)
print(result)

   total_rows  missing_company  missing_sector
0       49940                0               0


In [ ]:
query = """
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN Open_Price IS NULL THEN 1 ELSE 0 END) AS missing_open,
    SUM(CASE WHEN Close_Price IS NULL THEN 1 ELSE 0 END) AS missing_close,
    SUM(CASE WHEN Volume IS NULL THEN 1 ELSE 0 END) AS missing_volume
FROM Real_Time_Algorithmic_Market_Intelligence_Clean;
"""

result = pd.read_sql_query(query, conn)
print(result)

   total_rows  missing_open  missing_close  missing_volume
0       49940             0              0               0


In [ ]:
query = """
SELECT COUNT(*) AS invalid_price_rows
FROM Real_Time_Algorithmic_Market_Intelligence_Clean
WHERE High_Price < Low_Price
   OR High_Price < Open_Price
   OR High_Price < Close_Price
   OR Low_Price > Open_Price
   OR Low_Price > Close_Price;
"""

result = pd.read_sql_query(query, conn)
print(result)

   invalid_price_rows
0                 139


In [ ]:
query = """
SELECT
    Stock_Symbol,
    Trade_Date,
    Open_Price,
    High_Price,
    Low_Price,
    Close_Price
FROM Real_Time_Algorithmic_Market_Intelligence_Clean
WHERE High_Price < Low_Price
   OR High_Price < Open_Price
   OR High_Price < Close_Price
   OR Low_Price > Open_Price
   OR Low_Price > Close_Price
LIMIT 10;
"""

result = pd.read_sql_query(query, conn)
print(result)

  Stock_Symbol           Trade_Date  Open_Price  High_Price  Low_Price  \
0         SBIN  2024-06-17 00:00:00      732.14      748.55     721.96   
1   BHARTIARTL  2026-07-20 00:00:00     1568.37     1643.87    1593.59   
2           LT  2025-08-26 00:00:00     1568.37     3587.91    3461.75   
3    SUNPHARMA  2025-07-18 00:00:00     1856.88     1901.26    1847.34   
4   BHARTIARTL  2022-03-29 00:00:00     1538.35     1556.40    1519.94   
5         SBIN  2023-01-30 00:00:00      746.69      761.08     738.18   
6          ITC  2024-08-08 00:00:00     1568.37      504.66     492.13   
7        WIPRO  2025-03-13 00:00:00     1568.37      539.74     529.80   
8   TATAMOTORS  2026-03-26 00:00:00     1568.37      869.11     853.94   
9          TCS  2025-09-29 00:00:00     3528.01     3591.08    3450.17   

   Close_Price  
0      1567.93  
1      1613.93  
2      3520.80  
3      1567.93  
4      1567.93  
5      1567.93  
6       500.90  
7       533.68  
8       858.30  
9      1567.93 

In [ ]:
query = """
CREATE TABLE market_data_clean AS
SELECT *
FROM market_data_raw
WHERE High_Price >= Low_Price
  AND High_Price >= Open_Price
  AND High_Price >= Close_Price
  AND Low_Price <= Open_Price
  AND Low_Price <= Close_Price;
"""

conn.execute(query)
conn.commit()

In [ ]:
query = """
SELECT COUNT(*) AS total_clean_rows
FROM market_data_clean;
"""

result = pd.read_sql_query(query, conn)
print(result)

   total_clean_rows
0             49801


In [ ]:
query = """
SELECT COUNT(*) AS invalid_price_rows
FROM market_data_clean
WHERE High_Price < Low_Price
   OR High_Price < Open_Price
   OR High_Price < Close_Price
   OR Low_Price > Open_Price
   OR Low_Price > Close_Price;
"""

result = pd.read_sql_query(query, conn)
print(result)

   invalid_price_rows
0                   0


In [ ]:
query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT Stock_ID || Stock_Symbol || Trade_Date) AS unique_records
FROM market_data_clean;
"""

result = pd.read_sql_query(query, conn)
print(result)

   total_rows  unique_records
0       49801           49801


In [ ]:
query = """
SELECT
    Stock_ID,
    COUNT(*) AS record_count
FROM market_data_clean
GROUP BY Stock_ID
HAVING COUNT(*) > 1
ORDER BY record_count DESC;
"""

result = pd.read_sql_query(query, conn)
print(result)

Empty DataFrame
Columns: [Stock_ID, record_count]
Index: []


In [ ]:
query = """
SELECT
    Market,
    COUNT(*) AS total_records,
    AVG(Close_Price) AS avg_close_price,
    AVG(Daily_Return) AS avg_daily_return,
    AVG(Volatility) AS avg_volatility
FROM market_data_clean
GROUP BY Market
ORDER BY avg_daily_return DESC;
"""

result = pd.read_sql_query(query, conn)
print(result)

  Market  total_records  avg_close_price  avg_daily_return  avg_volatility
0    NSE          34865      2413.413721          0.003317        0.987584
1    BSE          14936      2397.884474         -0.003164        0.989527


In [ ]:
query = """
SELECT
    Sector,
    COUNT(*) AS total_records,
    AVG(Close_Price) AS avg_close_price,
    AVG(Daily_Return) AS avg_daily_return,
    AVG(Volatility) AS avg_volatility
FROM market_data_clean
GROUP BY Sector
ORDER BY avg_daily_return DESC;
"""

result = pd.read_sql_query(query, conn)
print(result)

           Sector  total_records  avg_close_price  avg_daily_return  \
0          Energy           3307      2804.494119          0.032004   
1  Infrastructure           3272      3497.233683          0.008166   
2      Automobile           6736      6473.341112          0.006749   
3         Banking          13174      1149.252172          0.004412   
4          Pharma           3186      1802.122081         -0.000354   
5         Telecom           3263      1600.379029         -0.005900   
6              IT          10036      1873.056676         -0.006477   
7            FMCG           6748      1561.676967         -0.010332   
8         Unknown             79      2488.565570         -0.159887   

   avg_volatility  
0        0.994501  
1        0.984961  
2        0.980937  
3        0.975955  
4        1.035138  
5        0.980620  
6        0.994858  
7        0.988660  
8        1.033599  


In [ ]:
query = """
SELECT
    Trading_Signal,
    COUNT(*) AS total_records,
    AVG(Daily_Return) AS avg_daily_return,
    AVG(Volatility) AS avg_volatility
FROM market_data_clean
GROUP BY Trading_Signal
ORDER BY total_records DESC;
"""

result = pd.read_sql_query(query, conn)
print(result)

  Trading_Signal  total_records  avg_daily_return  avg_volatility
0           HOLD          35029         -0.001263        0.555143
1            BUY           7416          1.544975        2.021927
2           SELL           7356         -1.542264        2.008015


In [ ]:
query = """
SELECT
    Stock_Symbol,
    COUNT(*) AS total_records,
    AVG(Daily_Return) AS avg_daily_return,
    AVG(Volatility) AS avg_volatility
FROM market_data_clean
GROUP BY Stock_Symbol
ORDER BY avg_daily_return DESC
LIMIT 10;
"""

result = pd.read_sql_query(query, conn)
print(result)

  Stock_Symbol  total_records  avg_daily_return  avg_volatility
0   TATAMOTORS           3359          0.033161        0.963497
1         SBIN           3343          0.032181        0.976894
2     RELIANCE           3316          0.029864        0.996053
3     HDFCBANK           3312          0.012106        0.958173
4           LT           3276          0.007639        0.984548
5         INFY           3321          0.007134        0.975434
6        WIPRO           3460          0.005480        1.004210
7   HINDUNILVR           3416          0.004293        0.981558
8    SUNPHARMA           3193          0.000752        1.035490
9   BHARTIARTL           3269         -0.007295        0.980877


In [ ]:
df_sql_clean = pd.read_sql_query("SELECT * FROM market_data_clean", conn)

df_sql_clean.to_csv('Real_Time_Algorithmic_Market_Intelligence_SQL_Clean.csv',index=False)

print(df_sql_clean.shape)

(49801, 15)


In [ ]:
from google.colab import files

files.download("Real_Time_Algorithmic_Market_Intelligence_SQL_Clean.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>